# 02. HCP Conversion

Build a file-map from `hcp_filt` and convert to `.pt` tensors.

This reproduces the nested output layout under:
`tensors/**/hcp_filt/<subject>/T1w/{rawavg.pt, aparc+aseg.pt}`.

In [ ]:
from pathlib import Path

import pandas as pd

from segmentation.openneuro import list_all_files, convert_file_map_to_tensors

In [ ]:
HCP_ROOT = Path("hcp_filt")
OUT_ROOT = Path("tensors")
N_JOBS = -1

if not HCP_ROOT.exists():
    raise FileNotFoundError(f"Missing HCP root: {HCP_ROOT.resolve()}")

files = list_all_files(HCP_ROOT)
files = [
    f
    for f in files
    if "T1w/" in f and (f.endswith("T1w_acpc_dc_restore.nii.gz") or f.endswith("aparc+aseg.nii.gz"))
]

file_map_hcp = {}
for f in files:
    parts = f.split("/")
    key = "/".join(parts[:-1])
    name = parts[-1].split(".")[0]

    if key not in file_map_hcp:
        file_map_hcp[key] = {"rawavg": None, "aparc+aseg": None}

    if name == "T1w_acpc_dc_restore":
        name = "rawavg"
    file_map_hcp[key][name] = f

file_map_hcp_complete = {
    k: v for k, v in file_map_hcp.items()
    if v["rawavg"] is not None and v["aparc+aseg"] is not None
}

display(pd.DataFrame([{
    "hcp_root": str(HCP_ROOT.resolve()),
    "out_root": str(OUT_ROOT.resolve()),
    "pairs_found": len(file_map_hcp),
    "pairs_complete": len(file_map_hcp_complete),
}]))


In [ ]:
results = convert_file_map_to_tensors(
    file_map_hcp_complete,
    out_root=str(OUT_ROOT),
    n_jobs=N_JOBS,
)

ok = sum(1 for r in results if bool(r.get("ok", False)))
failed = len(results) - ok

display(pd.DataFrame([{
    "converted_total": len(results),
    "converted_ok": ok,
    "converted_failed": failed,
}]))
